In [18]:
# ============================================================
# HIERARCHICAL SUMMARIZATION — LOAD + PREPROCESS SOURCE NOTES
#
# Use the same preprocessing as the other workflows so all
# summarization methods operate on the same source records.
# ============================================================

import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

from config.prompts import SYSTEM_PROMPT
from src.llm.llm import generate_summary



ModuleNotFoundError: No module named 'config'

In [4]:
PROJECT_ROOT = Path("../..").resolve()

NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes = pd.read_csv(NOTES_PATH)

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning:", len(notes_clean))
print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

Raw notes: 1602
After cleaning: 1595
After deduplication: 1103
Patients: 50


In [5]:
# ============================================================
# SANITY-TEST PATIENT
# ============================================================

SELECTED_PERSON_ID = (
    "c87e610e-ac2a-48cf-ac32-054f3e595498"
)

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"]
        == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Patient:", SELECTED_PERSON_ID)
print("Deduplicated notes:", len(patient_notes))

Patient: c87e610e-ac2a-48cf-ac32-054f3e595498
Deduplicated notes: 10


In [6]:
# ============================================================
# LEVEL 1 — NOTE-LEVEL SUMMARIZATION PROMPT
#
# Each source note is summarized independently before the
# patient-level synthesis step.
# ============================================================

NOTE_SUMMARY_PROMPT = """
You are summarizing one clinical note from a longitudinal patient record.

Create a concise clinical summary of this note.

Include only clinically meaningful information explicitly documented
in the note, such as:
- presenting symptoms or complaints
- diagnoses or clinical assessments
- examination findings
- investigations and results
- medications and treatments
- procedures or interventions
- clinically meaningful progression or response
- referrals, disposition, or follow-up plans

Rules:
- Do not add or infer information.
- Do not introduce new diagnoses, medications, or findings.
- Preserve clinically important values, doses, and findings.
- Preserve temporal information when explicitly stated.
- Remove administrative or repetitive wording unless clinically relevant.
- If the note contains very little clinical information, keep the summary brief.

Return only the note summary.
"""

In [13]:
# ============================================================
#
# This client goes DIRECTLY to OpenAI.
# ============================================================

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = "gpt-5.4-mini"

print("OpenAI client ready.")

print("Client class :", type(openai_client).__name__)
print("Client module:", type(openai_client).__module__)

OpenAI client ready.
Client class : OpenAI
Client module: openai


In [12]:
# ------------------------------------------------------------
# Generate one intermediate summary from one clinical note.
# ------------------------------------------------------------

def summarize_single_note(note_text):
    """
    Generate a concise clinical summary for one source note.
    """

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": NOTE_SUMMARY_PROMPT,
            },
            {
                "role": "user",
                "content": note_text,
            },
        ],
    )

    return response.choices[0].message.content

In [14]:
# ============================================================
# LEVEL 1 — GENERATE NOTE-LEVEL SUMMARIES
#
# Each completed note is checkpointed immediately so interrupted
# runs can resume without repeating successful LLM calls.
# ============================================================

HIERARCHICAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "hierarchical"
)

HIERARCHICAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTE_SUMMARY_PATH = (
    HIERARCHICAL_DIR
    / "sanity_note_summaries.json"
)

if NOTE_SUMMARY_PATH.exists():

    with NOTE_SUMMARY_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        note_summaries = json.load(file)

else:
    note_summaries = {}


for note_index, row in patient_notes.iterrows():

    note_key = str(note_index)

    if note_key in note_summaries:
        print(
            f"Skipping completed note: {note_index}"
        )
        continue

    print(
        f"Summarizing note "
        f"{note_index + 1}/{len(patient_notes)}"
    )

    summary = summarize_single_note(
        row["clean_note_text"]
    )

    note_summaries[note_key] = {
        "source_note_index": int(note_index),
        "creation_timestamp": str(
            row["creation_timestamp"]
        ),
        "summary": summary,
    }

    with NOTE_SUMMARY_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            note_summaries,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("Saved:", note_index)

    time.sleep(1)

Summarizing note 1/10
Saved: 0
Summarizing note 2/10
Saved: 1
Summarizing note 3/10
Saved: 2
Summarizing note 4/10
Saved: 3
Summarizing note 5/10
Saved: 4
Summarizing note 6/10
Saved: 5
Summarizing note 7/10
Saved: 6
Summarizing note 8/10
Saved: 7
Summarizing note 9/10
Saved: 8
Summarizing note 10/10
Saved: 9


In [15]:
# ============================================================
# BUILD CHRONOLOGICAL INTERMEDIATE CONTEXT
#
# The note-level summaries are placed back in their original
# chronological order before final patient-level synthesis.
# ============================================================

ordered_note_summaries = sorted(
    note_summaries.values(),
    key=lambda item: item[
        "source_note_index"
    ],
)

hierarchical_context_parts = []

for item in ordered_note_summaries:

    hierarchical_context_parts.append(
        f"[{item['creation_timestamp']}]\n"
        f"{item['summary']}"
    )

hierarchical_context = "\n\n".join(
    hierarchical_context_parts
)

print("Intermediate summaries:", len(ordered_note_summaries))
print("Context characters:", len(hierarchical_context))
print("Context words:", len(hierarchical_context.split()))

print("\n--- PREVIEW ---\n")
print(hierarchical_context[:3000])

Intermediate summaries: 10
Context characters: 5158
Context words: 785

--- PREVIEW ---

[04/01/2026 10:10]
1-year-old female with ear pain. Observations (HR, RR, temperature) were within normal limits, and she was in no acute distress. No allergies or past medical history documented. Directed to triage for further evaluation of ear pain.

[04/01/2026 10:30]
Afebrile with mild erythema around the right outer ear and moderate tenderness on palpation. No systemic signs of illness noted. Plan for doctor review and otoscopic examination to confirm diagnosis and determine treatment.

[04/01/2026 11:00]
Patient reviewed in the ED on 04/01/26. Baseline observations were normal: heart rate, temperature, and SpO2 all within normal limits. Patient was in no acute distress and remained stable. Plan was to await senior review; no medications or interventions were initiated, and the patient was to remain in the ED for further assessment.

[04/01/2026 11:30]
1-year-old female presented with irritabi

In [17]:
# ============================================================
# BUILD FINAL SYNTHESIS INPUT
#
# Convert the chronological note-level summaries into the same
# dataframe structure expected by generate_summary().
# ============================================================

hierarchical_notes_df = pd.DataFrame(
    [
        {
            "person_id": SELECTED_PERSON_ID,
            "creation_timestamp": item["creation_timestamp"],
            "clean_note_text": item["summary"],
        }
        for item in ordered_note_summaries
    ]
)

print("Intermediate summaries:", len(hierarchical_notes_df))

display(
    hierarchical_notes_df[
        [
            "creation_timestamp",
            "clean_note_text",
        ]
    ].head()
)

Intermediate summaries: 10


,creation_timestamp,clean_note_text
0,04/01/2026 10:10,1-year-old female with ear pain. Observations ...
1,04/01/2026 10:30,Afebrile with mild erythema around the right o...
2,04/01/2026 11:00,Patient reviewed in the ED on 04/01/26. Baseli...
3,04/01/2026 11:30,"1-year-old female presented with irritability,..."
4,04/01/2026 12:00,Acute right ear pain for 24 hours with intermi...


In [19]:
# ============================================================
# LEVEL 2 — FINAL HIERARCHICAL SYNTHESIS
#
# Use the same final SYSTEM_PROMPT used by the other workflows.
# ============================================================


hierarchical_summary = generate_summary(
    SELECTED_PERSON_ID,
    hierarchical_notes_df,
    SYSTEM_PROMPT,
)

print(hierarchical_summary)

NameError: name 'generate_summary' is not defined